**Setup (hidden in the slideshow)**

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as sla
from scipy.linalg import hilbert

plt.rcParams.update({"figure.dpi": 100, "font.size": 12})
rng = np.random.default_rng(1)

def arrow(ax, start, vec, color, label):
    """Draw a 2D arrow from `start` along `vec` and register a legend entry."""
    ax.annotate("", xy=start + vec, xytext=start,
                arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.plot([], [], color=color, lw=2, label=label)

def schur_snapshots(A, steps):
    """Remaining matrix (Schur complement embedded in n x n) after 0..steps elimination steps."""
    S = A.astype(float).copy()
    snaps = [S.copy()]
    for k in range(steps):
        S = S - np.outer(S[:, k], S[k, :]) / S[k, k]
        snaps.append(S.copy())
    return snaps

def wilkinson(n):
    """1 on the diagonal, -1 below it, last column of ones: worst case for partial pivoting."""
    W = np.eye(n) - np.tril(np.ones((n, n)), -1)
    W[:, -1] = 1.0
    return W

def augmented(A, alpha=1.0):
    """Augmented least-squares matrix [[0, A^T], [A, -alpha I]]."""
    m, n = A.shape
    return np.block([[np.zeros((n, n)), A.T],
                     [A, -alpha * np.eye(m)]])

# Lecture 5: Linear systems
### Geometry, elimination, and sensitivity

## Previous lecture
- Matrix rank
- Skeleton decomposition
- Low-rank approximation
- Singular Value Decomposition (SVD)

## Today

**Part I. Elimination**
- Two pictures of a linear system
- Gaussian elimination = LU decomposition = greedy rank-one peeling

**Part II. Stability**
- Pivoting: a good problem ruined by a bad algorithm
- Cholesky factorization
- Residual vs. error: a good algorithm cannot save a bad problem

**Part III. Sensitivity and least squares**
- Condition number and its geometry
- Least squares, pseudoinverse, QR

## Where linear systems come from

$Ax = f$ is the inner loop of almost everything:

- **Regression:** fitting a linear model leads to least squares.
- **Physics and engineering:** discretizing partial differential equations on a mesh with $N$ nodes gives $N\times N$ systems.
- **Optimization:** every Newton or Gauss–Newton step solves $H\,\Delta = -g$ ($H$ is the Hessian, the matrix of second derivatives; $g$ is the gradient), as do the KKT systems of constrained optimization.
- **Machine learning:** kernel ridge regression and Gaussian processes solve $(K + \lambda I)\alpha = y$, where $K$ is the kernel (similarity) matrix.

## Linear systems can be big

- Take a continuous problem, discretize it on a mesh with $N$ elements, and you get a linear system with an $N\times N$ matrix.
- Example of a mesh around an A319 aircraft (taken from the [GMSH website](http://geuz.org/gmsh/)).

<!-- Replace with the image path used in the original notebook -->
<img src="a319_4.png" width="500">

The main difficulty is that these systems are big: millions or billions of unknowns!

## Scales and structure

| Size | Typical storage |
|---|---|
| $n \lesssim 10^4$ | dense matrix fits in memory |
| $n \sim 10^4 - 10^6$ | sparse or structured |
| $n \sim 10^8 - 10^9$ | sparse + parallel computing |

A $100\times100\times100$ 3D grid gives $n = 10^6$ unknowns:
- the right-hand side takes **8 MB**;
- a dense matrix would take $10^{12}$ entries = **8 TB**.

## Real matrices are structured

Sparse, low-rank, Toeplitz (constant along diagonals), sparse in a certain basis, ...

The most famous one is the 1D discrete Laplacian:

$$
T = \begin{pmatrix}
2 & -1 & & & \\
-1 & 2 & -1 & & \\
 & \ddots & \ddots & \ddots & \\
 & & -1 & 2 & -1 \\
 & & & -1 & 2
\end{pmatrix}.
$$

- Storing it costs $O(n)$.
- Multiplying it by a vector costs $O(n)$.
- **But how do we solve a system with it?** Keep this matrix in mind; it will return.

## Two pictures of one system

$$
\begin{pmatrix} 2 & 1 \\ 1 & 3 \end{pmatrix}
\begin{pmatrix} x_1 \\ x_2 \end{pmatrix} =
\begin{pmatrix} 3 \\ 5 \end{pmatrix}
$$

- **Row picture.** Each equation is a line (in $\mathbb{R}^3$, a plane; in $\mathbb{R}^n$, a hyperplane). The solution is their intersection.
- **Column picture.** Find $x_1, x_2$ such that $x_1 a_1 + x_2 a_2 = f$, where $a_1, a_2$ are the columns of $A$. Solving a system means **expressing $f$ in the basis of the columns**.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
f = np.array([3.0, 5.0])
x = np.linalg.solve(A, f)          # exact answer: [0.8, 1.4]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
t = np.linspace(-1, 3, 100)
for i in range(2):
    ax[0].plot(t, (f[i] - A[i, 0] * t) / A[i, 1], label=f"equation {i + 1}")
ax[0].plot(*x, "ko", label="solution")
ax[0].set_title("Row picture: intersect lines")
ax[0].set_xlabel("$x_1$"); ax[0].set_ylabel("$x_2$"); ax[0].legend(); ax[0].grid(True)

arrow(ax[1], np.zeros(2), x[0] * A[:, 0], "C0", r"$x_1 a_1$")
arrow(ax[1], x[0] * A[:, 0], x[1] * A[:, 1], "C1", r"$x_2 a_2$")
arrow(ax[1], np.zeros(2), f, "k", r"$f$")
ax[1].set_xlim(-0.5, 4); ax[1].set_ylim(-0.5, 5.5)
ax[1].set_title("Column picture: combine columns to reach $f$")
ax[1].legend(); ax[1].grid(True)
plt.show()

> **Pause.** What does a *singular* matrix look like in each picture?

- **Rows:** parallel lines, so there is either no intersection or infinitely many points.
- **Columns:** $a_1, a_2$ lie on one line, so most $f$ are unreachable.

## Existence and uniqueness

For a **square** $A$ the following are equivalent:

- $Ax = f$ has a **unique solution for every** $f$;
- $\det A \neq 0$;
- $A$ has full rank (its columns form a basis).

If $A$ is singular, a solution exists only if $f \in \mathrm{range}(A)$ (the column space), and then there are infinitely many.

## Over- and underdetermined systems

- **Overdetermined** (more equations than unknowns): generically no solution, so we look for the best fit (least squares, Part III).
- **Underdetermined** (fewer equations than unknowns): infinitely many solutions, so we need an extra criterion, e.g. minimal norm.

## Forget Cramer's rule — and do not trust the determinant

- Cramer's rule needs $n+1$ determinants and is fine for $2\times 2$ only.
- There is a deeper point: **$\det A$ is a bad measure of how close to singular a matrix is.**

In [ ]:
n = 100
D = 0.1 * np.eye(n)
print("det(0.1 I):", np.linalg.det(D), "  cond:", np.linalg.cond(D))

B = np.array([[1.0, 1e8],
              [0.0, 1.0]])
print("det(B):    ", np.linalg.det(B), "  cond:", np.linalg.cond(B))

> **Pause.** $0.1I$ has a tiny determinant but is perfectly harmless. $B$ has determinant $1$ but is nearly singular.
> Can you find a tiny perturbation of $B$ that makes it singular?
> The right measure is the **condition number** (Part III).

## Gaussian elimination on a toy example

$$
\begin{aligned}
2x + 3y \phantom{{}+3z} &= 5,\\
2x \phantom{{}+3y} + 3z &= 5,\\
x + y \phantom{{}+3z} &= 2.
\end{aligned}
$$

From the third equation, $y = 2 - x$. Substituting into the first gives $2x + 6 - 3x = 5$, so $x = 1$, $y = 1$, and then $z = 1$.

This process — **Gaussian elimination** — is one of the most widely used algorithms.

## Forward and backward steps

- **Forward step.** Use equation 1 to express
  $$x_1 = \big(f_1 - (a_{12}x_2 + \dots + a_{1n}x_n)\big)/a_{11}$$
  and substitute it into equations $2,\dots,n$. Repeat with $x_2$, and so on.
  The numbers we divide by ($a_{11}$ and its successors) are called **pivots**; they must be nonzero.

- **Backward step.** The last equation contains only $x_n$. Solve it, substitute into the previous one, and so on, like falling dominoes.

## Elimination *is* a factorization

**LU decomposition:** $A = LU$, where
- $L$ is lower triangular (zeros strictly above the diagonal),
- $U$ is upper triangular (zeros strictly below the diagonal).

The factorization is unique if we require $\mathrm{diag}(L) = 1$ (unit lower triangular).

- $U$ is what remains after the forward step.
- $L$ stores the **multipliers** $\ell_{ik} = a_{ik}/a_{kk}$ used to eliminate.

## Why LU solves systems

$$ A^{-1}f = (LU)^{-1}f = U^{-1}L^{-1}f, $$

so $Ax = f$ is two triangular solves:

$$ Ly = f \quad (\text{forward substitution}), \qquad Ux = y \quad (\text{backward substitution}). $$

> **Pause.** Why is a triangular system easy? How many operations does it take?

Row $i$ needs about $i$ multiply-adds, so the total is about $n^2$ flops (floating-point operations). No search, no choices.

## The picture that matters: LU as rank-one peeling

One elimination step in block form, with pivot $a_{11}$ (scalar), column $c = A_{2:n,1}$, row $r^\top = A_{1,2:n}$:

$$
A = \begin{pmatrix} a_{11} & r^\top \\ c & A_{22} \end{pmatrix}
= \underbrace{\frac{1}{a_{11}}\begin{pmatrix} a_{11} \\ c\end{pmatrix}
\begin{pmatrix} a_{11} & r^\top\end{pmatrix}}_{\text{rank one: a "cross"}}
+ \begin{pmatrix} 0 & 0 \\ 0 & S \end{pmatrix},
\qquad S = A_{22} - \frac{c\, r^\top}{a_{11}}.
$$

$S$ is the **Schur complement**.

- One elimination step = subtract the rank-one "cross" through the pivot.
- This is **exactly the skeleton decomposition from the previous lecture**, built greedily, one cross at a time.
- $LU = \sum_{k=1}^n \ell_k u_k^\top$, where $\ell_k$ is the $k$-th column of $L$ and $u_k^\top$ the $k$-th row of $U$.

In [ ]:
def lu_nopivot(A):
    """LU without pivoting via rank-one peeling. A: (n, n) array. Returns L, U."""
    S = A.astype(float).copy()
    n = S.shape[0]
    L, U = np.zeros((n, n)), np.zeros((n, n))
    for k in range(n):
        L[:, k] = S[:, k] / S[k, k]      # column of the cross, scaled by the pivot
        U[k, :] = S[k, :]                # row of the cross
        S = S - np.outer(L[:, k], U[k, :])
    return L, U

n = 8
A = rng.standard_normal((n, n)) + n * np.eye(n)
L, U = lu_nopivot(A)
print("||LU - A|| =", np.linalg.norm(L @ U - A))

fig, axs = plt.subplots(1, 5, figsize=(15, 3))
for k, (ax, S) in enumerate(zip(axs, schur_snapshots(A, 4))):
    ax.imshow(np.abs(S), cmap="viridis")
    ax.set_title(f"after {k} steps"); ax.axis("off")
plt.suptitle("|remaining matrix|: each step removes one row and one column")
plt.show()

> **Pause.** Let $A$ have rank $r$ (and nonzero pivots). What happens to the Schur complement after $r$ steps? Predict, then run.

In [ ]:
Uf, Vf = rng.standard_normal((n, 3)), rng.standard_normal((n, 3))
A_low = Uf @ Vf.T                                  # rank 3
for k, S in enumerate(schur_snapshots(A_low, 4)):
    print(f"step {k}: ||S|| = {np.linalg.norm(S):.2e}")

The remainder vanishes after $r$ steps: **elimination discovers the rank.** Rank-revealing factorizations and cross approximation build on this idea.

## Cost

- Step $k$ updates an $(n-k)\times(n-k)$ block, so the total is $\sum_k 2(n-k)^2 \approx \frac{2}{3}n^3$ flops.
- Each triangular solve costs $\approx n^2$ flops.

**Factor once in $O(n^3)$, then solve for many right-hand sides in $O(n^2)$ each.**

> **Pause.** Can Strassen's algorithm help here?

## Block LU

$$
\begin{pmatrix} A_{11} & A_{12} \\ A_{21} & A_{22} \end{pmatrix}
= \begin{pmatrix} L_{11} & 0 \\ L_{21} & L_{22} \end{pmatrix}
\begin{pmatrix} U_{11} & U_{12} \\ 0 & U_{22} \end{pmatrix}
$$

$$
A_{11} = L_{11}U_{11},\quad U_{12} = L_{11}^{-1}A_{12},\quad L_{21} = A_{21}U_{11}^{-1},\quad
A_{22} - L_{21}U_{12} = L_{22}U_{22}.
$$

- Only half-size LUs, triangular solves and matrix products are needed. So **yes, in theory:** if matrix multiplication costs $O(n^{\omega})$, LU costs $O(n^{\omega})$ too.
- **In practice** the main gain of blocking is different: most of the work becomes matrix–matrix products (BLAS-3), which are very fast on modern hardware.

## When does LU exist?

Let $A_k$ denote the leading $k\times k$ submatrix (first $k$ rows and columns).

- **Definition.** $A$ is **strictly regular** if all leading submatrices $A_k$ are nonsingular.
- **Theorem.** If $\det A_k \neq 0$ for $k = 1,\dots,n-1$, then $A = LU$ with unit $L$ exists and is unique.
- The pivots are ratios of minors: $u_{kk} = \det A_k / \det A_{k-1}$, since $\det A_k = u_{11}\cdots u_{kk}$.
- **Converse (for nonsingular $A$):** if $A = LU$, then all $\det A_k \neq 0$. The converse fails for singular matrices: the zero matrix has $0 = I\cdot 0$.

## Uniqueness

**Claim.** If $L$ is unit lower triangular, the LU decomposition of a nonsingular matrix is unique.

**Proof.** $L_1U_1 = L_2U_2$ implies $L_2^{-1}L_1 = U_2U_1^{-1}$.
- The left side is unit lower triangular.
- The right side is upper triangular.
- So both equal $I$: $L_1 = L_2$, $U_1 = U_2$. $\blacksquare$

> **Pause.** $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ is as nice as a matrix can be: orthogonal, with condition number 1.
> Does it have an LU decomposition? What is the minimal fix?

No: the first pivot is $0$. The fix is to **swap the rows**. This leads to pivoting in Part II.

## Back to the Laplacian: factors vs. inverse

> **Predict.** $T$ is tridiagonal. What do the nonzero patterns of $L$, $U$ and $T^{-1}$ look like?

In [ ]:
n = 50
T = 2 * np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)
L, U = lu_nopivot(T)
Tinv = np.linalg.inv(T)

fig, axs = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, M, name in zip(axs[:4], [T, L, U, Tinv], ["T", "L", "U", "T^{-1}"]):
    ax.spy(np.abs(M) > 1e-12, markersize=2); ax.set_title(f"nonzeros of ${name}$")
for j in [10, 25, 40]:
    axs[4].plot(Tinv[:, j], label=f"column {j}")
axs[4].set_title("columns of $T^{-1}$"); axs[4].legend(); axs[4].grid(True)
plt.show()

## What we see

- $L$ and $U$ stay **bidiagonal**: LU preserves the band.
- $T^{-1}$ is **completely dense**. Its columns are "tents": $T$ is a discrete string, and column $j$ of $T^{-1}$ is the string's shape when you push it at point $j$ (the Green's function). A local push moves the whole string.
- **Moral:** the inverse is a non-local object. **Store factors, not inverses.** For $n = 10^6$, the factors of $T$ take a few MB, while $T^{-1}$ takes 8 TB.

# Break

Next: **Part II. Stability**

## When LU fails: a good problem, a bad algorithm

$$
A = \begin{pmatrix}\varepsilon & 1\\ 1 & 1\end{pmatrix},\qquad
f = \begin{pmatrix}1\\2\end{pmatrix},\qquad
x \approx \begin{pmatrix}1\\1\end{pmatrix}.
$$

- The condition number of this matrix is about $2.6$, so the problem is perfectly fine.
- The matrix is strictly regular for any $\varepsilon \ne 0$, so LU exists.

> **Predict.** What happens with $\varepsilon = 10^{-17}$?

In [ ]:
eps = 1e-17
A = np.array([[eps, 1.0], [1.0, 1.0]])
f = np.array([1.0, 2.0])

L, U = lu_nopivot(A)
print("L =\n", L, "\nU =\n", U)
print("LU - A =\n", L @ U - A)

y = sla.solve_triangular(L, f, lower=True)
x_bad = sla.solve_triangular(U, y)
print("no pivoting  :", x_bad)
print("with pivoting:", np.linalg.solve(A, f))

## What happened

- The multiplier is $1/\varepsilon = 10^{17}$.
- $u_{22} = 1 - 10^{17}$ rounds to $-10^{17}$: the entry $a_{22} = 1$ is **swamped**, and its information is gone.
- Then $x_1 = (1 - x_2)/\varepsilon$ divides a cancellation error by a tiny number.

**Fix: partial (row) pivoting.** At step $k$, swap rows so that the entry largest in modulus in column $k$ becomes the pivot:

$$ A = PLU, \qquad P \text{ a permutation matrix},\qquad |\ell_{ij}| \le 1. $$

## Pivoting is not bulletproof: the growth factor

The multipliers are bounded, but entries of $U$ can still grow.

- **Growth factor** $\rho = \max|u_{ij}| / \max|a_{ij}|$.
- With partial pivoting, $\rho \le 2^{n-1}$, and the bound is attained.

> **Pause.** Can you construct a matrix where the entries of $U$ double at every step?

The Wilkinson matrix: $1$ on the diagonal, $-1$ below it, and a last column of ones.

In [ ]:
for n in [10, 20, 40, 60]:
    W = wilkinson(n)
    P, L, U = sla.lu(W)                             # scipy convention: W = P L U
    x_true = np.ones(n)
    x = np.linalg.solve(W, W @ x_true)
    print(f"n={n:3d}  growth={np.abs(U).max():.1e}  cond(W)={np.linalg.cond(W):.1e}  "
          f"error={np.linalg.norm(x - x_true) / np.linalg.norm(x_true):.1e}")

growth = {}
for n in [50, 100, 200, 400]:
    G = rng.standard_normal((n, n))
    _, _, U = sla.lu(G)
    growth[n] = round(np.abs(U).max() / np.abs(G).max(), 1)
print("random matrices:", growth)

> **Pause.** The Wilkinson matrix is well conditioned, yet LU with pivoting produces garbage for large $n$. Random matrices show tiny growth.
> Why does the worst case essentially never happen in practice?

This was an open puzzle for decades. One explanation is **smoothed analysis** (Sankar, Spielman, Teng): tiny random perturbations destroy the bad examples.

## Positive definite matrices

**Definition.** $A$ is **positive definite** if $(Ax, x) > 0$ for all $x \neq 0$.
- Over $\mathbb{C}^n$ this forces $A = A^*$ (Hermitian).
- Over $\mathbb{R}^n$, $A$ may be non-symmetric, e.g. $\begin{pmatrix}1&1\\-1&1\end{pmatrix}$: only the symmetric part $\tfrac12(A + A^\top)$ matters.

**Picture.** $x^*Ax$ is an "energy bowl": a paraboloid with elliptic level sets, curving upward in every direction.

Examples: covariance matrices, Gram matrices, kernel matrices, Hessians at a strict minimum.

## Cholesky factorization

**Theorem.** If $A$ is Hermitian positive definite (HPD), then
- every leading submatrix is HPD (restrict $x$ to the first $k$ coordinates), so $A$ is strictly regular;
- $A = LL^*$, where $L$ is lower triangular with a positive diagonal.

It is sometimes called the "square root" of a matrix. It takes half the work of LU ($\approx \tfrac13 n^3$ flops) and **needs no pivoting**.

## Why Cholesky is stable, in one line

Compare the diagonal entries of $A = LL^*$:

$$ a_{ii} = \sum_{j} |\ell_{ij}|^2 \quad\Longrightarrow\quad |\ell_{ij}| \le \sqrt{a_{ii}}. $$

- No entry of the factor can grow beyond the square root of the diagonal of $A$, so there is nothing to amplify rounding errors.
- Equivalently, the Schur complement of an HPD matrix is again HPD, and its diagonal can only decrease.

**Practical trick.** The cheapest way to *test* positive definiteness is to try `np.linalg.cholesky`: it fails iff the matrix is not (numerically) positive definite.

## Machine learning view: sampling from a Gaussian

To sample from $\mathcal{N}(0, \Sigma)$ ($\Sigma$ is the covariance matrix):

if $\Sigma = LL^\top$ and $z \sim \mathcal{N}(0, I)$, then $Lz \sim \mathcal{N}(0, \Sigma)$, since $\mathrm{Cov}(Lz) = L\,I\,L^\top = \Sigma$.

Gaussian processes, Kalman filters and diffusion models use this.

In [ ]:
Sigma = np.array([[2.0, 1.2], [1.2, 1.0]])
Lc = np.linalg.cholesky(Sigma)
Z = rng.standard_normal((2, 2000))
X = Lc @ Z
plt.figure(figsize=(5, 5))
plt.scatter(*Z, s=2, alpha=0.3, label="z ~ N(0, I)")
plt.scatter(*X, s=2, alpha=0.3, label="L z ~ N(0, Σ)")
plt.axis("equal"); plt.legend(); plt.grid(True)
plt.title("The Cholesky factor turns a round cloud into an ellipse")
plt.show()

## A good algorithm, a bad problem

Take the **Hilbert matrix** $H_{ij} = 1/(i+j-1)$. We **manufacture** a solution $x_{\text{true}} = (1,\dots,1)^\top$, set $f = Hx_{\text{true}}$, and solve.

Two quantities to track:
- **residual** $\|Hx - f\|/\|f\|$: does $x$ satisfy the equations?
- **error** $\|x - x_{\text{true}}\|/\|x_{\text{true}}\|$: is $x$ the right answer?

> **Predict first:** how do they behave as $n$ grows?

In [ ]:
ns = list(range(2, 16))
res, err, bnd = [], [], []
for n in ns:
    H = hilbert(n)
    x_true = np.ones(n)
    f = H @ x_true
    x = np.linalg.solve(H, f)
    res.append(np.linalg.norm(H @ x - f) / np.linalg.norm(f))
    err.append(np.linalg.norm(x - x_true) / np.linalg.norm(x_true))
    bnd.append(np.linalg.cond(H) * np.finfo(float).eps)

plt.figure(figsize=(7, 4.5))
plt.semilogy(ns, res, "o-", label="relative residual")
plt.semilogy(ns, err, "s-", label="relative error")
plt.semilogy(ns, bnd, "k--", label=r"cond$(H)\cdot \varepsilon_{\rm mach}$")
plt.xlabel("n"); plt.legend(); plt.grid(True)
plt.title("A small residual does NOT mean an accurate solution")
plt.show()

## What we see

- The residual stays near machine precision: the algorithm did its job. $x$ is the **exact** solution of a slightly perturbed system $(H + \Delta H)x = f$ with $\|\Delta H\|/\|H\| \sim 10^{-16}$. This property is called **backward stability**.
- The error explodes: the **problem itself** amplifies tiny perturbations. That amplification factor is the **condition number**.
- This is not the algorithm's fault. Already when $H$ is stored in memory its entries are rounded, so no algorithm can do better.

## Two sources of error

$$ \text{forward error} \;\lesssim\; \underbrace{\text{condition number}}_{\text{problem}} \times \underbrace{\text{backward error}}_{\text{algorithm}}. $$

| Example | Problem | Algorithm |
|---|---|---|
| $\begin{pmatrix}\varepsilon & 1\\1&1\end{pmatrix}$ without pivoting | stable | **unstable** |
| Hilbert matrix with pivoting | **unstable** | stable |

To understand the condition number, we first need the inverse.

# Break

Next: **Part III. Sensitivity and least squares**

## The inverse

$A^{-1}$ is the matrix $X$ with $AX = XA = I$.

- Column $i$ of $X$ solves $Ax_i = e_i$ ($e_i$ is the $i$-th column of $I$).
- So computing $A^{-1}$ means solving $n$ systems with one LU factorization.
- If we have $A^{-1}$, then $x = A^{-1}f$, since $A(A^{-1}f) = (AA^{-1})f = f$.

But we have just seen that $A^{-1}$ can be dense when $A$ is sparse. In practice we use $A^{-1}$ as a **theoretical tool**, not as something to compute.

## Neumann series

**Theorem.** If $\|F\| < 1$ in a submultiplicative norm ($\|AB\| \le \|A\|\|B\|$), then $I - F$ is invertible and

$$ (I - F)^{-1} = I + F + F^2 + \dots = \sum_{k=0}^{\infty} F^k. $$

This is the matrix geometric series.

**Proof.** $(I-F)\sum_{k=0}^N F^k = I - F^{N+1}$ and $\|F^{N+1}\| \le \|F\|^{N+1} \to 0$.

**Norm of the inverse.** $\left\|\sum_{k=0}^N F^k\right\| \le \sum_{k=0}^N \|F\|^k \|I\| \le \frac{\|I\|}{1-\|F\|}$, and $\|I\| = 1$ for induced norms.

> **Pause.** Is $\|F\| < 1$ *necessary*? Take $F = \begin{pmatrix}0 & 2\\0&0\end{pmatrix}$, with $\|F\| = 2$. Compute $F^2$. What is the sum of the series?

$F^2 = 0$, so the series is just $I + F$, and indeed $(I - F)(I + F) = I$.

- The series converges iff $\|F^k\| \to 0$.
- The right quantity is $\lim_{k\to\infty} \|F^k\|^{1/k}$, the **spectral radius** (next lecture); it does not depend on the norm.
- But the norms can first *grow*:

In [ ]:
F = np.array([[0.9, 1.0], [0.0, 0.9]])
ks = np.arange(60)
norms = [np.linalg.norm(np.linalg.matrix_power(F, k), 2) for k in ks]
plt.figure(figsize=(7, 4))
plt.semilogy(ks, norms, "o-"); plt.axhline(1, color="k", lw=0.8)
plt.xlabel("k"); plt.ylabel(r"$\|F^k\|_2$"); plt.grid(True)
plt.title("The 'hump': ||F|| > 1, yet the series converges")
plt.show()

## Perturbation of the inverse

If $\|A^{-1}E\| < 1$, then

$$ (A+E)^{-1} = (I + A^{-1}E)^{-1}A^{-1} = \sum_{k\ge0}(-A^{-1}E)^kA^{-1}, $$

and therefore

$$ \frac{\|(A+E)^{-1} - A^{-1}\|}{\|A^{-1}\|} \le \frac{\|A^{-1}\|\,\|E\|}{1 - \|A^{-1}E\|}. $$

The **norm of the inverse** controls the sensitivity.

## The condition number

For the perturbed system $(A + \Delta A)\hat x = f + \Delta f$:

$$
\frac{\|\hat x - x\|}{\|x\|} \le \frac{\mathrm{cond}(A)}{1 - \mathrm{cond}(A)\frac{\|\Delta A\|}{\|A\|}}
\left(\frac{\|\Delta A\|}{\|A\|} + \frac{\|\Delta f\|}{\|f\|}\right),
\qquad \mathrm{cond}(A) = \|A\|\,\|A^{-1}\|.
$$

- To first order: **relative error out ≈ cond(A) × relative error in.**
- **Rule of thumb:** with $\mathrm{cond}(A) \approx 10^k$ you lose about $k$ of your 16 decimal digits.
- The condition number depends on the norm. In the 2-norm, $\mathrm{cond}_2(A) = \sigma_{\max}/\sigma_{\min}$, because $\|A\|_2 = \sigma_{\max}$ and the singular values of $A^{-1}$ are $1/\sigma_i$.
- If $\Delta A = 0$: $\ \dfrac{\|\hat x - x\|}{\|x\|} \le \mathrm{cond}(A)\dfrac{\|\Delta f\|}{\|f\|}$.

## Derivation (for reference)

$$
\begin{aligned}
\hat x - x &= (A + \Delta A)^{-1} (f + \Delta f) - A^{-1} f \\
&= \Big[\sum_{k=1}^{\infty} (-A^{-1} \Delta A)^k\Big] A^{-1} f + \Big[\sum_{k=0}^{\infty} (-A^{-1} \Delta A)^k \Big] A^{-1} \Delta f,
\end{aligned}
$$

so

$$
\frac{\|\hat x - x\|}{\|x\|} \le \frac{1}{1 - \|A^{-1}\Delta A\|}\left(\|A^{-1}\|\|\Delta A\| + \frac{\|A^{-1}\|\|\Delta f\|}{\|x\|}\right).
$$

Finally, $\|f\| = \|Ax\| \le \|A\|\|x\|$ gives $\frac{1}{\|x\|} \le \frac{\|A\|}{\|f\|}$, and $\|A^{-1}\Delta A\| \le \mathrm{cond}(A)\frac{\|\Delta A\|}{\|A\|}$.

## Geometry of the condition number

- $A$ maps the unit circle to an ellipse with semi-axes $\sigma_1 u_1$ and $\sigma_2 u_2$ ($u_i$ are the left singular vectors).
- $A^{-1}$ maps a small ball of uncertainty around $f$ to an ellipse stretched by $1/\sigma_{\min}$ along the "weak" direction.
- **$\mathrm{cond}_2(A)$ is the eccentricity of this ellipse.**

In [ ]:
A = np.array([[1.0, 0.9], [0.9, 1.0]])        # sigma = 1.9 and 0.1
theta = np.linspace(0, 2 * np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])
f0 = np.array([1.0, 1.0])
r = 0.05
Ainv = np.linalg.inv(A)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].plot(*(f0[:, None] + r * circle), "C1"); ax[0].plot(*f0, "ko")
ax[0].set_title(r"data space: $f$ known up to $\|\Delta f\| \leq 0.05$")
x0 = Ainv @ f0
ax[1].plot(*(x0[:, None] + r * Ainv @ circle), "C1"); ax[1].plot(*x0, "ko")
ax[1].set_title(f"solution space: cond = {np.linalg.cond(A):.0f}")
for a in ax:
    a.set_aspect("equal"); a.grid(True)
plt.show()

## The same in the row picture: fuzzy lines

If the right-hand side is noisy, each line becomes a band. Nearly parallel lines intersect over a long, thin region.

In [ ]:
x_true = np.array([1.0, 1.0])
mats = [(np.array([[1.0, 1.0], [1.0, -1.0]]), "well-conditioned"),
        (np.array([[1.0, 1.0], [1.0, 1.05]]), "ill-conditioned")]
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
t = np.linspace(-3, 5, 100)
for k, (M, name) in enumerate(mats):
    f0 = M @ x_true
    sols = np.array([np.linalg.solve(M, f0 + 0.05 * rng.standard_normal(2)) for _ in range(500)])
    for i in range(2):
        ax[k].plot(t, (f0[i] - M[i, 0] * t) / M[i, 1], "k-", lw=1)
    ax[k].scatter(sols[:, 0], sols[:, 1], s=3, alpha=0.4)
    ax[k].set_xlim(-3, 5); ax[k].set_ylim(-3, 5); ax[k].grid(True)
    ax[k].set_title(f"{name}, cond = {np.linalg.cond(M):.0f}")
plt.show()

## Distance to singularity

By the Eckart–Young theorem (previous lecture), the closest singular matrix to $A$ in the 2-norm is at distance $\sigma_{\min}$:

$$ \min\left\{\frac{\|\Delta A\|_2}{\|A\|_2} : A + \Delta A \text{ singular}\right\} = \frac{1}{\mathrm{cond}_2(A)}. $$

**An ill-conditioned matrix is one that is relatively close to a singular one.** This is the correct replacement for "small determinant".

> **Pause.** Return to $B = \begin{pmatrix}1 & 10^8\\0&1\end{pmatrix}$. Find a perturbation of size about $10^{-8}$ that makes it singular.

Put $-10^{-8}$ in the bottom-left corner: $\det = 1 - 10^{-8}\cdot 10^{8} = 0$.

## Worst case vs. typical case

The bound $\mathrm{cond}(A)$ is the *worst case*. Using the SVD $A = \sum_i \sigma_i u_i v_i^*$,

$$ x = A^{-1}f = \sum_i \frac{u_i^* f}{\sigma_i}\, v_i, $$

so **the component of $f$ along $u_i$ is amplified by $1/\sigma_i$**.

What actually happens depends on how $f$ (and its noise) is spread over the singular directions. Let us compare $f = $ ones and a random $f$ for a Hilbert-like matrix.

In [ ]:
n = 40
idx = np.arange(n)
A = 1.0 / (idx[:, None] + idx[None, :] + 0.5)   # Hilbert-like (Cauchy) matrix
U_, s, Vt = np.linalg.svd(A)
rhs_list = [(np.ones(n), "f = ones"), (rng.standard_normal(n), "f = random")]

plt.figure(figsize=(7, 4.5))
plt.semilogy(s, "k--", label=r"$\sigma_i$")
for rhs, name in rhs_list:
    plt.semilogy(np.abs(U_.T @ rhs), "o", ms=4, label=rf"$|u_i^* f|$, {name}")
    x = np.linalg.solve(A, rhs)
    print(f"{name:11s}: ||x|| = {np.linalg.norm(x):.1e}")
plt.xlabel("i"); plt.legend(); plt.grid(True)
plt.title("Picard plot: how f is spread over singular directions")
plt.show()

> **Pause.** Which right-hand side gives a larger $\|x\|$, and why? What does the "floor" at $\sim 10^{-16}$ mean, and what happens to it after division by $\sigma_i$?

- Smooth data live in the leading singular directions.
- Noise (including rounding) is spread evenly and gets amplified most in the trailing directions, by up to $1/\sigma_{\min}$.
- This is why we *regularize* — see the pseudoinverse below.

## Overdetermined systems: least squares

With $A \in \mathbb{R}^{m\times n}$, $m > n$ (more data than parameters), there is generically no exact solution. Minimize the residual instead:

$$ \|Ax - b\|_2 \to \min_x . $$

The simplest example you all know is **linear fitting**: fitting a set of 2D points with a line.

## Geometry: least squares = projection

- $Ax$ runs over the column space of $A$ (a subspace, $n$-dimensional if the columns are independent).
- The best $Ax$ is the **orthogonal projection** of $b$ onto it.
- So the residual $r = b - Ax$ is orthogonal to every column:

$$ A^*(b - Ax) = 0 \quad\Longleftrightarrow\quad A^*Ax = A^*b \quad(\text{normal equations}). $$

- $A^*A$ is the **Gram matrix** (pairwise inner products of the columns).
- The same equation follows from setting the gradient $\nabla\|Ax-b\|_2^2 = 2(A^*Ax - A^*b)$ to zero.

## Example: fitting a line

Model $y = ax + b$, noisy data $(x_1, y_1), \dots, (x_m, y_m)$:

$$
\begin{pmatrix} x_1 & 1 \\ \vdots & \vdots \\ x_m & 1 \end{pmatrix}
\begin{pmatrix} a \\ b \end{pmatrix} \approx
\begin{pmatrix} y_1 \\ \vdots \\ y_m \end{pmatrix}.
$$

Least squares minimizes the sum of squared **vertical** distances.

In [ ]:
rng_ls = np.random.default_rng(42)
a_true, b_true = 1.0, 2.0
m = 10
xi = np.arange(m, dtype=float)
yi = a_true * xi + b_true + 2 * rng_ls.standard_normal(m)

M = np.column_stack([xi, np.ones(m)])          # columns: x_i and 1
coef, *_ = np.linalg.lstsq(M, yi, rcond=None)  # coef = [a, b]
fit = M @ coef

plt.figure(figsize=(7, 4.5))
plt.plot(xi, yi, "o", label="data $(x_i, y_i)$")
plt.plot(xi, fit, label=f"least squares: y = {coef[0]:.2f} x + {coef[1]:.2f}")
plt.vlines(xi, fit, yi, colors="gray", linestyles=":", label="residuals")
plt.legend(); plt.grid(True); plt.show()

> **Pause.** Why vertical distances and not perpendicular ones? Which assumption about the noise is hidden here?

- The noise is assumed to be in $y$ only, while $x$ is exact.
- If both coordinates are noisy, perpendicular distances lead to **total least squares**, and that problem is solved by the SVD.

## Why not the normal equations?

$\mathrm{cond}_2(A^*A) = \mathrm{cond}_2(A)^2$, because the singular values of $A^*A$ are $\sigma_i^2$.

**Forming $A^*A$ squares the condition number**, so half of the digits are gone before you even start solving.

> **Predict.** For a degree-11 polynomial fit, how many correct digits will each method give?

In [ ]:
m, n = 100, 12
t = np.linspace(0, 1, m)
A = np.vander(t, n, increasing=True)     # polynomial fit: columns 1, t, ..., t^{n-1}
x_true = np.ones(n)
b = A @ x_true

x_ne = np.linalg.solve(A.T @ A, A.T @ b)
x_ls = np.linalg.lstsq(A, b, rcond=None)[0]
print(f"cond(A) = {np.linalg.cond(A):.1e},  cond(A^T A) = {np.linalg.cond(A.T @ A):.1e}")
print(f"error, normal equations : {np.linalg.norm(x_ne - x_true):.1e}")
print(f"error, lstsq (SVD-based): {np.linalg.norm(x_ls - x_true):.1e}")

## Pseudoinverse

If the columns are dependent, $A^*A$ is singular. Define the **Moore–Penrose pseudoinverse**

$$ A^\dagger = \lim_{\alpha\to 0^+}(A^*A + \alpha I)^{-1}A^*. $$

- Full column rank: $A^\dagger = (A^*A)^{-1}A^*$.
- Square nonsingular: $A^\dagger = (A^*A)^{-1}A^* = A^{-1}A^{-*}A^* = A^{-1}$.
- $x = A^\dagger b$ is the least-squares solution of **minimal Euclidean norm**.

## Pseudoinverse via the SVD

With $A = U\Sigma V^*$: $\ A^\dagger = V\Sigma^\dagger U^*$, where $\Sigma^\dagger$ inverts the nonzero $\sigma_i$ and keeps the zeros. Indeed,

$$
(A^*A+\alpha I)^{-1}A^* = V(\Sigma^2+\alpha I)^{-1}\Sigma U^*,
\qquad \frac{\sigma_i}{\sigma_i^2 + \alpha} \xrightarrow{\alpha\to0} \begin{cases}1/\sigma_i, & \sigma_i > 0,\\ 0, & \sigma_i = 0.\end{cases}
$$

For rectangular $A$, $\mathrm{cond}_2(A) = \sigma_{\max}/\sigma_{\min}$, the ratio of the largest and the smallest nonzero singular values.

## Machine learning view: do not take the limit

A fixed $\alpha > 0$ is **ridge regression** (Tikhonov regularization). It multiplies each $1/\sigma_i$ by a *filter factor*

$$ \frac{\sigma_i^2}{\sigma_i^2 + \alpha}: $$

- directions with $\sigma_i \gg \sqrt{\alpha}$ are kept;
- directions with $\sigma_i \ll \sqrt\alpha$, where noise dominates (recall the Picard plot), are suppressed.

**Truncated SVD** uses a hard 0/1 filter instead.

In [ ]:
sig = np.logspace(-4, 1, 300)
plt.figure(figsize=(7, 4))
for alpha in [1e-6, 1e-4, 1e-2]:
    plt.semilogx(sig, sig**2 / (sig**2 + alpha), label=rf"ridge, $\alpha$={alpha:g}")
plt.xlabel(r"$\sigma$"); plt.ylabel("filter factor"); plt.legend(); plt.grid(True)
plt.title("Ridge regression = soft spectral filter")
plt.show()

## The canonical way: QR

Any $A$ with full column rank factors as $A = QR$, where
- $Q \in \mathbb{C}^{m\times n}$ has **orthonormal columns** ($Q^*Q = I$),
- $R$ is upper triangular (details in a later lecture).

Then

$$ x = (A^*A)^{-1}A^*b = (R^*Q^*QR)^{-1}R^*Q^*b = R^{-1}Q^*b, $$

so we solve the triangular system $Rx = Q^*b$ in $O(n^2)$ operations.

Here $\mathrm{cond}(R) = \mathrm{cond}(A)$, not its square.

## Augmented system

Introduce the residual $r = Ax - b$ as an unknown. Then $A^*r = 0$ and

$$
\begin{pmatrix} 0 & A^* \\ A & -I\end{pmatrix}\begin{pmatrix}x\\ r\end{pmatrix} = \begin{pmatrix}0\\ b\end{pmatrix}.
$$

The matrix is square ($m+n$), symmetric, sparse if $A$ is, and we never form $A^*A$.

> **Pause.** Is its condition number equal to $\mathrm{cond}(A)$?

On each pair of singular vectors $(v_i, u_i)$ the matrix acts as $\begin{pmatrix}0&\sigma_i\\\sigma_i&-1\end{pmatrix}$, with eigenvalues $\lambda = \tfrac{-1\pm\sqrt{1+4\sigma_i^2}}{2}$.

- For small $\sigma$, $\lambda \approx \sigma^2$, so the condition number behaves like $\max(\sigma_{\max},1)/\sigma_{\min}^2$ — **not** $\mathrm{cond}(A)$!
- The fix is to scale: replace $-I$ by $-\alpha I$ with $\alpha \sim \sigma_{\min}$, which gives a condition number of about $\mathrm{cond}(A)$ times a small constant.

In [ ]:
A = np.vander(np.linspace(0, 1, 50), 8, increasing=True)
smin = np.linalg.svd(A, compute_uv=False)[-1]
print(f"cond(A)                     = {np.linalg.cond(A):.2e}")
print(f"cond(augmented, alpha=1)    = {np.linalg.cond(augmented(A, 1.0)):.2e}")
print(f"cond(augmented, alpha=smin) = {np.linalg.cond(augmented(A, smin)):.2e}")

## Summary

- Two pictures: **rows** = intersecting hyperplanes, **columns** = expressing $f$ in the column basis.
- Gaussian elimination = LU = **greedy rank-one peeling** (skeleton decomposition, one cross at a time). Cost: $\tfrac23 n^3$ to factor, $O(n^2)$ per solve.
- Factors preserve sparsity; inverses destroy it. **Do not form $A^{-1}$.**
- Small pivots break LU, so use **pivoting** ($A = PLU$). The growth factor can be $2^{n-1}$ but practically never is.
- **Cholesky** for HPD matrices needs no pivoting, and its entries cannot grow.

## Summary (continued)

- **Residual ≠ error.** Forward error ≲ condition number × backward error.
- $\mathrm{cond}_2(A) = \sigma_{\max}/\sigma_{\min}$ = eccentricity of the SVD ellipse = $1/$(relative distance to singularity). The determinant measures none of this.
- Least squares = orthogonal projection. The normal equations square the condition number; use QR or SVD.
- Ridge regression = a soft spectral filter.
- Without structure, we can solve systems with up to about $10^4$ unknowns on a laptop (memory limits).

## Questions to think about

1. LU is the greedy skeleton decomposition. Is it the *best* rank-$r$ approximation after $r$ steps? How would you choose pivots to make it better?
2. Is `inv(A) @ b` really worse than `solve(A, b)`? Compare cost, residual and error experimentally.
3. Construct a $2\times2$ matrix with $\det = 10^{10}$ and $\mathrm{cond}_2 = 10^{10}$.
4. Why does the Cholesky factorization of a covariance matrix fail in floating point for nearly duplicated features? What does adding a small "jitter" $\delta I$ (as in Gaussian process code) do to the singular values?

## Next lecture
- Eigenvalues and eigenvectors
- Schur theorem